In [ ]:
"""
# Object-Based Defect Detection Pipeline (Simplified)

This notebook demonstrates object-level defect detection pipeline - no whole image classification.
"""

# Import all modules with correct paths
import sys
import os

# Add the abbvisionsystem directory to the Python path
sys.path.append('abbvisionsystem')
sys.path.append('abbvisionsystem/model_training')
sys.path.append('abbvisionsystem/models')

# Now import the modules
from abbvisionsystem.model_training.data_manager import DataManager, SyntheticDataGenerator
from abbvisionsystem.model_training.model_trainer import DefectClassificationTrainer, AnomalyDetectionTrainer
from abbvisionsystem.models.defect_detection_model import ObjectDefectDetectionModel
from abbvisionsystem.model_training.visualization import Visualizer
from abbvisionsystem.model_training.object_detector import ObjectDetector

import logging
import os

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration
CONFIG = {
    'source_dir': 'data/choco-pie',
    'object_dataset': 'object_detection_dataset',
    'model_name': 'object_defect_classifier',
    'epochs': 50,
    'batch_size': 32,
    'image_size': (224, 224)
}

print("🚀 Starting Object-Based Defect Detection Pipeline")
print("=" * 60)
print("📌 This pipeline ONLY trains object-level detection models")
print("❌ No whole image classification - objects are detected individually")
print("=" * 60)

# Step 1: Data Organization for Object Detection
print("\n📁 Step 1: Organizing Data for Object Detection")
data_manager = DataManager(CONFIG['source_dir'], CONFIG['object_dataset'])

print("Organizing for object-level detection...")
print("📂 Processing folders: good/, defect/, both/")
try:
    metadata = data_manager.organize_object_detection_data()
    print(f"✅ Object detection data organized successfully")
    print(f"📊 Created metadata with {len(metadata)} object entries")
except Exception as e:
    print(f"❌ Error organizing object detection data: {str(e)}")
    exit(1)

# Verify object detection directory
object_dir = f"{CONFIG['object_dataset']}_objects"
print(f"\n📂 Object detection directory: {object_dir}")
if os.path.exists(object_dir):
    print(f"✅ Object detection directory exists")
    for split in ['train', 'validation', 'test']:
        split_dir = os.path.join(object_dir, split)
        if os.path.exists(split_dir):
            normal_count = len([f for f in os.listdir(os.path.join(split_dir, 'normal')) 
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            defect_count = len([f for f in os.listdir(os.path.join(split_dir, 'defect')) 
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            print(f"   {split}: {normal_count} normal objects, {defect_count} defect objects")
else:
    print(f"❌ Object detection directory does not exist")
    exit(1)

# Step 2: Data Visualization
print("\n🎨 Step 2: Data Visualization")
visualizer = Visualizer()
try:
    visualizer.visualize_object_extraction(dataset_dir=object_dir)
    print("✅ Visualization completed")
except Exception as e:
    print(f"⚠️ Visualization error: {str(e)}")

# Step 3: Train Object Detection Model
print("\n🏋️ Step 3: Training Object Detection Model")
print("🎯 Training classifier for individual objects only")

train_dir = os.path.join(object_dir, 'train')
if os.path.exists(train_dir):
    normal_files = [f for f in os.listdir(os.path.join(train_dir, 'normal')) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    defect_files = [f for f in os.listdir(os.path.join(train_dir, 'defect')) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    print(f"📊 Training data: {len(normal_files)} normal objects, {len(defect_files)} defect objects")
    
    if len(normal_files) > 0 and len(defect_files) > 0:
        try:
            trainer = DefectClassificationTrainer(
                data_dir=object_dir,
                model_name=CONFIG['model_name'],
                image_size=CONFIG['image_size']
            )
            
            model = trainer.create_model()
            print(f"✅ Model created with {model.count_params():,} parameters")
            
            # Train model
            print("🚀 Starting training...")
            history = trainer.train(epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'])
            
            # Evaluate model
            print("\n📊 Step 4: Model Evaluation")
            results = trainer.evaluate()
            
            # Plot training history
            trainer.plot_training_history()
            
            # Save model
            print("\n💾 Step 5: Saving Model")
            trainer.save_model()
            
            print("✅ Object detection model trained and saved!")
            
        except Exception as e:
            print(f"❌ Error training model: {str(e)}")
            exit(1)
    else:
        print("⚠️ Insufficient training data found")
        print(f"   Normal objects: {len(normal_files)}")
        print(f"   Defect objects: {len(defect_files)}")
        print("💡 Need at least 1 sample of each class")
        exit(1)
else:
    print(f"❌ Training directory not found: {train_dir}")
    exit(1)

# Step 6: Test Object Detection Model
print("\n🧪 Step 6: Testing Object Detection Model")
object_detection_model = ObjectDefectDetectionModel(
    classifier_path=f"trained_models/{CONFIG['model_name']}.h5"
)

if object_detection_model.load():
    print("✅ Object detection model loaded successfully")
    
    # Test on various image categories including 'both'
    test_base_dir = CONFIG['source_dir']
    test_categories = ['good', 'defect', 'both']
    
    for category in test_categories:
        test_dir = os.path.join(test_base_dir, category)
        if os.path.exists(test_dir):
            print(f"\n🔍 Testing on {category} images:")
            
            test_files = [f for f in os.listdir(test_dir)[:3] 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            
            for img_file in test_files:
                try:
                    import cv2
                    img_path = os.path.join(test_dir, img_file)
                    img = cv2.imread(img_path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        
                        detections = object_detection_model.predict(img)
                        print(f"   📸 {img_file}: {detections['num_detections']} objects detected")
                        
                        if detections['num_detections'] > 0:
                            # Visualize and save results
                            result_img = object_detection_model.visualize_detections(img, detections)
                            result_path = f"object_detection_result_{category}_{img_file}"
                            cv2.imwrite(result_path, cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
                            print(f"      💾 Saved result to {result_path}")
                            
                            # Print detection details
                            for i, detail in enumerate(detections.get('object_details', [])):
                                class_id = detections['classes'][i]
                                confidence = detections['scores'][i]
                                class_name = object_detection_model.categories.get(class_id, {}).get('name', f'Class {class_id}')
                                print(f"      🎯 Object {i+1}: {class_name} ({confidence:.2%})")
                        else:
                            print(f"      ⚠️ No objects detected in {img_file}")
                            
                except Exception as e:
                    print(f"      ❌ Error testing on {img_file}: {str(e)}")
        else:
            print(f"   ⚠️ Test directory not found: {test_dir}")
else:
    print("❌ Failed to load object detection model")
    exit(1)

# Step 7: Validation with 'both' folder
print("\n🔍 Step 7: Special Validation with 'both' folder")
both_dir = os.path.join(CONFIG['source_dir'], 'both')
if os.path.exists(both_dir):
    print("📂 Testing mixed object images (both good and defective objects):")
    
    both_files = [f for f in os.listdir(both_dir) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    for img_file in both_files:
        try:
            img_path = os.path.join(both_dir, img_file)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                detections = object_detection_model.predict(img)
                num_objects = detections['num_detections']
                
                if num_objects > 0:
                    # Count normal vs defective
                    normal_count = sum(1 for i in range(num_objects) if detections['classes'][i] == 0)
                    defect_count = sum(1 for i in range(num_objects) if detections['classes'][i] == 1)
                    
                    print(f"   📸 {img_file}:")
                    print(f"      🎯 Total objects: {num_objects}")
                    print(f"      ✅ Normal: {normal_count}")
                    print(f"      ❌ Defects: {defect_count}")
                    
                    # Save detailed result
                    result_img = object_detection_model.visualize_detections(img, detections, threshold=0.3)
                    result_path = f"mixed_detection_result_{img_file}"
                    cv2.imwrite(result_path, cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
                    print(f"      💾 Detailed result saved to {result_path}")
                    
        except Exception as e:
            print(f"      ❌ Error processing {img_file}: {str(e)}")
else:
    print("   ⚠️ 'both' folder not found - skipping mixed validation")

# Final Summary
print("\n🎉 Object Detection Pipeline Complete!")
print("=" * 60)
print("✅ Object detection data organized")
print("✅ Object detection model trained and saved")
print("✅ Model validation completed")
print("✅ Testing on all categories completed")
print("✅ Mixed object validation completed")

# Model Summary
if os.path.exists("trained_models"):
    models = [f for f in os.listdir("trained_models") if f.endswith(('.h5', '.keras'))]
    print(f"\n📄 {len(models)} models saved in 'trained_models/' directory:")
    for model_file in models:
        print(f"   🤖 {model_file}")
        
    if f"{CONFIG['model_name']}.h5" in models:
        print("\n🎯 Object detection model is ready for deployment!")
        print("💡 Launch the Streamlit app to use the trained model:")
        print("   streamlit run abbvisionsystem/app.py")
    else:
        print("\n⚠️ Expected model file not found")
else:
    print("\n❌ No trained_models directory found")

print("\n📋 Next Steps:")
print("1. Launch Streamlit app: streamlit run abbvisionsystem/app.py")
print("2. Test object detection on new images")
print("3. Use 'Individual Objects' mode for object-level classification")
print("4. Deploy to production environment")
print("5. Monitor model performance and retrain as needed")

print("\n🎯 Model Capabilities:")
print("✅ Detects individual objects in images")
print("✅ Classifies each object as Normal or Defective")
print("✅ Works with multiple objects per image")
print("✅ Provides confidence scores and bounding boxes")
print("❌ Does not do whole image classification")